# 01 EDA

Phase 6 notebook for universe coverage, return distribution, and missingness diagnostics.

This notebook is read-only on existing artifacts.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from src.config import PROCESSED_DIR

processed = Path(PROCESSED_DIR)
panel = pd.read_parquet(processed / 'features_panel.parquet')
panel['date'] = pd.to_datetime(panel['date'])
feature_cols = [c for c in panel.columns if c not in ['permno', 'date', 'ret_exc']]
print(f'Rows: {len(panel):,}')
print(f'Months: {panel["date"].nunique():,}')
print(f'Features: {len(feature_cols)}')

In [ ]:
panel['year'] = panel['date'].dt.year
universe = panel.groupby('year')['permno'].nunique()

fig, ax = plt.subplots(figsize=(10, 4))
universe.plot(ax=ax, color='#1f77b4', lw=2)
ax.set_title('Universe Coverage: Number of Stocks per Year')
ax.set_xlabel('Year')
ax.set_ylabel('N stocks')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(panel['ret_exc'].dropna(), bins=120, color='#2ca02c', alpha=0.85)
ax.set_title('Distribution of Excess Returns (ret_exc)')
ax.set_xlabel('Monthly excess return')
ax.set_ylabel('Count')
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

print(panel['ret_exc'].describe().to_string())

In [ ]:
missing_by_year = panel.groupby('year')[feature_cols].apply(lambda d: d.isna().mean())
top_missing = missing_by_year.mean().sort_values(ascending=False).head(25).index.tolist()
heat = missing_by_year[top_missing].T

fig, ax = plt.subplots(figsize=(12, 8))
im = ax.imshow(heat.values, aspect='auto', interpolation='nearest')
ax.set_title('Missingness Heatmap by Year (Top 25 Features by Missing Rate)')
ax.set_xticks(range(len(heat.columns)))
ax.set_xticklabels(heat.columns, rotation=90)
ax.set_yticks(range(len(heat.index)))
ax.set_yticklabels(heat.index)
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Missing share')
plt.tight_layout()
plt.show()